# Credit Card Fraud Detection Using Machine Learning

**Objective:** Build a binary classification model to identify potentially fraudulent credit-card transactions.

This notebook follows a reproducible workflow: data loading → exploratory analysis → missing-value treatment → correlation analysis → categorical encoding → class-imbalance handling → XGBoost training → model evaluation.

> **Dataset note:** The raw Parquet dataset is intentionally excluded from GitHub because of its size. Place `ieee_fraud_detection.parquet` in the repository's `data/` directory before running the notebook.

## 1. Import Libraries

The libraries below cover data manipulation, visualisation, preprocessing, class-imbalance handling, and model training.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

## 2. Load the Dataset

The original notebook used a Google Drive path. For GitHub, the project uses a relative path so that another user can reproduce the analysis after placing the dataset in `data/`.

In [ ]:
DATA_PATH = Path("../data/ieee_fraud_detection.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place ieee_fraud_detection.parquet in the repository's data/ folder."
    )

df = pd.read_parquet(DATA_PATH, engine="pyarrow")
print(f"Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

## 3. Initial Data Inspection

Before preprocessing, inspect the first records, dimensions, data types, summary statistics, and target variable.

In [ ]:
display(df.head())
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.value_counts())
print("\nSummary statistics:")
display(df.describe(include="all").T.head(20))

## 4. Set Transaction ID as the Index

`TransactionID` identifies a transaction rather than describing its fraud characteristics. It is therefore used as the dataframe index and is not passed to the model as a predictor.

In [ ]:
if "TransactionID" in df.columns:
    df = df.set_index("TransactionID")

display(df.head())

## 5. Missing-Value Analysis

Fraud-detection datasets often contain substantial missingness. The helper function below reports both the number and percentage of missing values for a selected data type.

In [ ]:
def missing_values(dataframe, dtypes="object"):
    """Return missing-value counts and percentages for selected data types."""
    total_rows = len(dataframe)
    missing_count = dataframe.select_dtypes(include=dtypes).isna().sum()
    missing_percent = (missing_count / total_rows) * 100

    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent.round(2),
    })

    return summary[summary["missing_count"] > 0].sort_values(
        "missing_percent", ascending=False
    )

print("Numerical columns with missing values:")
display(missing_values(df, dtypes="number").head(20))

print("Categorical columns with missing values:")
display(missing_values(df, dtypes=["object", "category"]))

## 6. Handle Missing Values

For the baseline model:

- Numerical features are imputed with their median.
- Categorical features are imputed with their most frequent value (mode).

This keeps the baseline workflow simple and avoids dropping large numbers of observations.

In [ ]:
# Numerical features: replace missing values with the feature median.
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical features: replace missing values with the most frequent category.
cat_cols = df.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    mode = df[col].mode(dropna=True)
    if not mode.empty:
        df[col] = df[col].fillna(mode.iloc[0])

remaining_missing = int(df.isna().sum().sum())
print(f"Remaining missing values after imputation: {remaining_missing:,}")

## 7. Correlation Analysis

Highly correlated numerical variables can contain redundant information. The function below identifies pairs with an absolute correlation above 0.90.

> **Important:** The original project identified highly correlated features but did not drop them in the final run. This notebook keeps that behaviour so the baseline modelling approach remains comparable.

In [ ]:
corr_matrix = df.select_dtypes(include="number").corr()

def get_high_correlation(correlation_matrix, threshold=0.90):
    """Return feature pairs whose absolute correlation exceeds a threshold."""
    upper = correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )

    high_corr = []
    for col in upper.columns:
        for row in upper.index:
            value = upper.loc[row, col]
            if pd.notna(value) and abs(value) > threshold:
                high_corr.append((col, row, value))

    return sorted(high_corr, key=lambda x: abs(x[2]), reverse=True)

high_corr_pairs = get_high_correlation(corr_matrix, threshold=0.90)
print(f"Number of highly correlated feature pairs: {len(high_corr_pairs)}")

for feature_1, feature_2, correlation in high_corr_pairs[:20]:
    print(f"{feature_1} <-> {feature_2} = {correlation:.2f}")

## 8. Categorical Feature Encoding

The dataset contains a small number of categorical variables. `P_emaildomain` has relatively high cardinality, while `ProductCD`, `card4`, `card6`, and `M6` have fewer categories.

The baseline project uses label encoding for `P_emaildomain` and one-hot encoding for the lower-cardinality variables.

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
cardinality = df[categorical_columns].nunique().sort_values(ascending=False)
print("Categorical feature cardinality:")
display(cardinality)

# Label encode the higher-cardinality email-domain feature.
label_encoder = LabelEncoder()
if "P_emaildomain" in df.columns:
    df["P_emaildomain"] = label_encoder.fit_transform(df["P_emaildomain"].astype(str))

# One-hot encode low-cardinality categorical variables.
one_hot_columns = [
    col for col in ["ProductCD", "card4", "card6", "M6"]
    if col in df.columns
]

df = pd.get_dummies(df, columns=one_hot_columns, drop_first=True, dtype=int)

print(f"Feature matrix after encoding: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

## 9. Analyse Class Imbalance

Fraud is the minority class. This is critical because a model can achieve high accuracy simply by predicting most transactions as legitimate.

For this reason, precision, recall, F1-score and the confusion matrix are reported alongside accuracy.

In [ ]:
class_distribution = df["isFraud"].value_counts().sort_index()
class_percentage = df["isFraud"].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    "count": class_distribution,
    "percentage": class_percentage.round(2),
})
summary.index = ["Legitimate (0)", "Fraud (1)"]
display(summary)

plt.figure(figsize=(6, 4))
sns.countplot(x="isFraud", data=df)
plt.title("Distribution of Fraudulent Transactions")
plt.xlabel("Transaction class")
plt.ylabel("Number of transactions")
plt.xticks([0, 1], ["Not Fraud", "Fraud"])
plt.tight_layout()
plt.show()

## 10. Prepare Training and Test Data

The target is `isFraud`. An 80/20 train-test split is used with a fixed random seed for reproducibility.

In [ ]:
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).rename("percentage").mul(100).round(2))

## 11. Build the Imbalanced-Learning Pipeline

The pipeline performs three steps during model fitting:

1. **StandardScaler** scales numerical model inputs.
2. **SMOTE** creates synthetic minority-class examples using only the training data.
3. **XGBoost** learns the fraud-classification model.

Keeping SMOTE inside the pipeline prevents the test set from being oversampled.

In [ ]:
classifier = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
)

pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", classifier),
])

pipeline

## 12. Train the XGBoost Model

The model is fitted using the training data. SMOTE is applied internally by the pipeline during this step.

In [ ]:
pipeline.fit(X_train, y_train)
print("Model training completed.")

## 13. Generate Predictions

Predictions are generated on the untouched test set.

In [ ]:
y_pred = pipeline.predict(X_test)
print("Predictions generated successfully.")

## 14. Evaluate Model Performance

Because this is an imbalanced fraud-detection problem, the classification report is more informative than accuracy alone.

- **Precision:** Of transactions predicted as fraud, how many were actually fraud?
- **Recall:** Of all actual fraud transactions, how many did the model identify?
- **F1-score:** Harmonic mean of precision and recall.

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Fraud", "Fraud"],
    digits=2,
))

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

## 15. Confusion Matrix

The confusion matrix shows where the model makes correct predictions and where it produces false positives or false negatives.

For fraud detection, false negatives are particularly important because they represent fraudulent transactions that the model fails to flag.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ["Not Fraud", "Fraud"]

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 16. Results and Interpretation

The original notebook run produced approximately **97% accuracy**. For the fraud class, it reported **0.69 precision, 0.52 recall, and 0.59 F1-score**.

The key limitation is fraud recall: a recall of 0.52 means a substantial proportion of fraudulent transactions were not detected in that evaluation. In a real deployment, the classification threshold and business cost of false negatives versus false positives would need to be considered.

## 17. Next Steps

Potential improvements for a future version include:

- Tune XGBoost hyperparameters with cross-validation.
- Evaluate Precision-Recall AUC and ROC-AUC.
- Optimise the probability threshold instead of relying on the default 0.50 threshold.
- Compare XGBoost with LightGBM, CatBoost, Random Forest and Logistic Regression.
- Investigate feature importance and SHAP explanations.
- Consider temporal validation because transaction data has a time component.
- Add model calibration and monitoring for a production-style workflow.